In [ ]:
# Import Library
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

import matplotlib.pyplot as plt

In [ ]:
# up dataset
from google.colab import files
uploaded = files.upload()

Saving PhiUSIIL_Phishing_URL_Dataset.csv to PhiUSIIL_Phishing_URL_Dataset.csv


In [ ]:
# Load CSV
import pandas as pd

df = pd.read_csv('PhiUSIIL_Phishing_URL_Dataset.csv')
df.head()

,FILENAME,URL,URLLength,Domain,DomainLength,IsDomainIP,TLD,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,...,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label
0,521848.txt,https://www.southbankmosaics.com,31,www.southbankmosaics.com,24,0,com,100.0,1.000000,0.522907,...,0,0,1,34,20,28,119,0,124,1
1,31372.txt,https://www.uni-mainz.de,23,www.uni-mainz.de,16,0,de,100.0,0.666667,0.032650,...,0,0,1,50,9,8,39,0,217,1
2,597387.txt,https://www.voicefmradio.co.uk,29,www.voicefmradio.co.uk,22,0,uk,100.0,0.866667,0.028555,...,0,0,1,10,2,7,42,2,5,1
3,554095.txt,https://www.sfnmjournal.com,26,www.sfnmjournal.com,19,0,com,100.0,1.000000,0.522907,...,1,1,1,3,27,15,22,1,31,1
4,151578.txt,https://www.rewildingargentina.org,33,www.rewildingargentina.org,26,0,org,100.0,1.000000,0.079963,...,1,0,1,244,15,34,72,1,85,1


In [ ]:
# di sini buat ngehapus duplicate url
print("Before removing duplicate URLs:", df.shape)
print("Duplicate URLs:", df['URL'].duplicated().sum())

df = df.drop_duplicates(subset='URL', keep='first').reset_index(drop=True)

print("After removing duplicate URLs:", df.shape)
print("Duplicate URLs:", df['URL'].duplicated().sum())

Before removing duplicate URLs: (235795, 56)
Duplicate URLs: 425
After removing duplicate URLs: (235370, 56)
Duplicate URLs: 0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 235370 entries, 0 to 235369
Data columns (total 56 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   FILENAME                    235370 non-null  object 
 1   URL                         235370 non-null  object 
 2   URLLength                   235370 non-null  int64  
 3   Domain                      235370 non-null  object 
 4   DomainLength                235370 non-null  int64  
 5   IsDomainIP                  235370 non-null  int64  
 6   TLD                         235370 non-null  object 
 7   URLSimilarityIndex          235370 non-null  float64
 8   CharContinuationRate        235370 non-null  float64
 9   TLDLegitimateProb           235370 non-null  float64
 10  URLCharProb                 235370 non-null  float64
 11  TLDLength                   235370 non-null  int64  
 12  NoOfSubDomain               235370 non-null  int64  
 13  HasObfuscation

In [ ]:
df.shape

(235370, 56)

In [ ]:
df.columns

Index(['FILENAME', 'URL', 'URLLength', 'Domain', 'DomainLength', 'IsDomainIP',
       'TLD', 'URLSimilarityIndex', 'CharContinuationRate',
       'TLDLegitimateProb', 'URLCharProb', 'TLDLength', 'NoOfSubDomain',
       'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio',
       'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL',
       'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL',
       'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL',
       'SpacialCharRatioInURL', 'IsHTTPS', 'LineOfCode', 'LargestLineLength',
       'HasTitle', 'Title', 'DomainTitleMatchScore', 'URLTitleMatchScore',
       'HasFavicon', 'Robots', 'IsResponsive', 'NoOfURLRedirect',
       'NoOfSelfRedirect', 'HasDescription', 'NoOfPopup', 'NoOfiFrame',
       'HasExternalFormSubmit', 'HasSocialNet', 'HasSubmitButton',
       'HasHiddenFields', 'HasPasswordField', 'Bank', 'Pay', 'Crypto',
       'HasCopyrightInfo', 'NoOfImage', 'NoOfCSS', 'NoOfJS', 'NoOfSelfRef',
       'NoOfEmptyRef', 'NoOf

In [ ]:
df['label'].value_counts()

,count
label,
1,134850
0,100520


In [ ]:
# preprocessing buat menghapus kolom teks, X dan y
drop_cols = ['FILENAME', 'URL', 'Domain', 'TLD', 'Title']

X = df.drop(columns=drop_cols + ['label'])
y = df['label']

print("Jumlah fitur input:", X.shape[1])
print("Jumlah data:", X.shape[0])
print(y.value_counts())

Jumlah fitur input: 50
Jumlah data: 235370
label
1    134850
0    100520
Name: count, dtype: int64


In [ ]:
# split train,validation,test
from sklearn.model_selection import train_test_split

# 70% training, 30% sementara
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# 30% sementara dibagi dua, 15% validation dan 15% testing
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Train:", X_train.shape, y_train.value_counts().to_dict())
print("Validation:", X_val.shape, y_val.value_counts().to_dict())
print("Test:", X_test.shape, y_test.value_counts().to_dict())

Train: (164759, 50) {1: 94395, 0: 70364}
Validation: (35305, 50) {1: 20227, 0: 15078}
Test: (35306, 50) {1: 20228, 0: 15078}


In [ ]:
# balancing train, validation, dan test menggunakan random under sampler
from imblearn.under_sampling import RandomUnderSampler

rus_train = RandomUnderSampler(random_state=42)
rus_val = RandomUnderSampler(random_state=42)
rus_test = RandomUnderSampler(random_state=42)

X_train_balanced, y_train_balanced = rus_train.fit_resample(
    X_train, y_train
)

X_val_balanced, y_val_balanced = rus_val.fit_resample(
    X_val, y_val
)

X_test_balanced, y_test_balanced = rus_test.fit_resample(
    X_test, y_test
)

print("Training setelah balancing:")
print(y_train_balanced.value_counts())

print("\nValidation setelah balancing:")
print(y_val_balanced.value_counts())

print("\nTesting setelah balancing:")
print(y_test_balanced.value_counts())

Training setelah balancing:
label
0    70364
1    70364
Name: count, dtype: int64

Validation setelah balancing:
label
0    15078
1    15078
Name: count, dtype: int64

Testing setelah balancing:
label
0    15078
1    15078
Name: count, dtype: int64


In [ ]:
# ngedefinisikan 5 model
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=5000,
            random_state=42
        ))
    ]),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        random_state=42,
        eval_metric="logloss"
    ),

    "AdaBoost": AdaBoostClassifier(
        random_state=42
    )
}

print("Model yang digunakan:")
for name in models:
    print("-", name)

Model yang digunakan:
- Logistic Regression
- Decision Tree
- Random Forest
- XGBoost
- AdaBoost


In [ ]:
# Training, evaluasi, dan pengukuran waktu
import time
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

results = []

for name, model in models.items():

    # mengukur waktu training
    train_start = time.perf_counter()
    model.fit(X_train_balanced, y_train_balanced)
    train_time = time.perf_counter() - train_start

    # prediksi validation
    y_val_pred = model.predict(X_val_balanced)

    # ini buat gukur waktu prediksi test beberapa kali supaya lebih stabil
    prediction_times = []

    for _ in range(10):
        predict_start = time.perf_counter()
        y_test_pred = model.predict(X_test_balanced)
        prediction_times.append(
            time.perf_counter() - predict_start
        )

    average_prediction_time = np.mean(prediction_times)

    # waktu prediksi rata-rata untuk satu data
    inference_ms_per_sample = (
        average_prediction_time / len(X_test_balanced)
    ) * 1000

    results.append({
        "Model": name,

        "Val Accuracy": accuracy_score(
            y_val_balanced, y_val_pred
        ),
        "Val Precision": precision_score(
            y_val_balanced,
            y_val_pred,
            average="weighted"
        ),
        "Val Recall": recall_score(
            y_val_balanced,
            y_val_pred,
            average="weighted"
        ),
        "Val F1-score": f1_score(
            y_val_balanced,
            y_val_pred,
            average="weighted"
        ),

        "Test Accuracy": accuracy_score(
            y_test_balanced, y_test_pred
        ),
        "Test Precision": precision_score(
            y_test_balanced,
            y_test_pred,
            average="weighted"
        ),
        "Test Recall": recall_score(
            y_test_balanced,
            y_test_pred,
            average="weighted"
        ),
        "Test F1-score": f1_score(
            y_test_balanced,
            y_test_pred,
            average="weighted"
        ),

        "Training Time (s)": train_time,
        "Prediction Time (s)": average_prediction_time,
        "Inference (ms/sample)": inference_ms_per_sample
    })

results_df = pd.DataFrame(results)

results_df.sort_values(
    by=["Test Accuracy", "Inference (ms/sample)"],
    ascending=[False, True]
)

,Model,Val Accuracy,Val Precision,Val Recall,Val F1-score,Test Accuracy,Test Precision,Test Recall,Test F1-score,Training Time (s),Prediction Time (s),Inference (ms/sample)
1,Decision Tree,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.517223,0.007107,0.000236
3,XGBoost,0.999967,0.999967,0.999967,0.999967,1.000000,1.000000,1.000000,1.000000,1.906417,0.040487,0.001343
2,Random Forest,0.999967,0.999967,0.999967,0.999967,1.000000,1.000000,1.000000,1.000000,13.789173,0.112209,0.003721
4,AdaBoost,0.999967,0.999967,0.999967,0.999967,1.000000,1.000000,1.000000,1.000000,12.919975,0.155438,0.005154
0,Logistic Regression,0.999934,0.999934,0.999934,0.999934,0.999901,0.999901,0.999901,0.999901,0.780885,0.015598,0.000517


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

for name, model in models.items():
    y_test_pred = model.predict(X_test_balanced)

    print("=" * 50)
    print(name)

    print("Confusion Matrix:")
    print(confusion_matrix(y_test_balanced, y_test_pred))

    print("\nClassification Report:")
    print(
        classification_report(
            y_test_balanced,
            y_test_pred,
            digits=6
        )
    )

Logistic Regression
Confusion Matrix:
[[15075     3]
 [    0 15078]]

Classification Report:
              precision    recall  f1-score   support

           0   1.000000  0.999801  0.999901     15078
           1   0.999801  1.000000  0.999901     15078

    accuracy                       0.999901     30156
   macro avg   0.999901  0.999901  0.999901     30156
weighted avg   0.999901  0.999901  0.999901     30156

Decision Tree
Confusion Matrix:
[[15078     0]
 [    0 15078]]

Classification Report:
              precision    recall  f1-score   support

           0   1.000000  1.000000  1.000000     15078
           1   1.000000  1.000000  1.000000     15078

    accuracy                       1.000000     30156
   macro avg   1.000000  1.000000  1.000000     30156
weighted avg   1.000000  1.000000  1.000000     30156

Random Forest
Confusion Matrix:
[[15078     0]
 [    0 15078]]

Classification Report:
              precision    recall  f1-score   support

           0   1.000000 

In [ ]:
# ngecek overlap url nya
train_urls = set(df.loc[X_train.index, 'URL'])
val_urls = set(df.loc[X_val.index, 'URL'])
test_urls = set(df.loc[X_test.index, 'URL'])

print("Train-Val overlap:", len(train_urls.intersection(val_urls)))
print("Train-Test overlap:", len(train_urls.intersection(test_urls)))
print("Val-Test overlap:", len(val_urls.intersection(test_urls)))

Train-Val overlap: 0
Train-Test overlap: 0
Val-Test overlap: 0


In [ ]:
# feature importance
rf_model = models["Random Forest"]

feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

feature_importance.head(10)

,Feature,Importance
3,URLSimilarityIndex,0.189058
49,NoOfExternalRef,0.166164
22,LineOfCode,0.149689
47,NoOfSelfRef,0.114116
44,NoOfImage,0.087081
46,NoOfJS,0.064887
36,HasSocialNet,0.048779
43,HasCopyrightInfo,0.026040
32,HasDescription,0.021650
23,LargestLineLength,0.017658
